# Chess Board Analysis
## Classical computer vision pipeline for board reading, piece detection, and classification

**Course:** INE410121 / TRV410001 - Computer Vision - UFSC
**Dataset:** Synthetic Chess Board Images - Kaggle (thefamousrat)

# Introduction

This notebook presents a **complete Classical Computer Vision pipeline** applied to the problem of detecting and analysing chess boards. The goal is to demonstrate how traditional image-processing techniques can be combined to:

1. **Detect** a chessboard in an image;
2. **Correct the perspective** to obtain a top-down view;
3. **Segment** the board into an 8x8 grid;
4. **Identify** which squares are occupied by pieces;
5. **Detect moves** by comparing two board states.

The chess scenario is chosen because of the richness of classical techniques it requires: edge detection, Hough transform for lines, contour detection, perspective correction via homography, thresholding, morphological operations and visual-feature analysis. **No Deep Learning technique or pre-trained model is used.**

# Problem Definition

**Given**: A rendered image of a chess board with pieces, photographed from an angular perspective.

**Goal**: Automatically determine the board state - which squares (A1–H8) are occupied - using exclusively classical computer vision techniques.

## Real-world applications

- **Game digitisation**: convert physical games to digital notation (PGN);
- **Analysis assistants**: feed engines with the position detected by camera;
- **Accessibility**: automatic description of games;
- **CV education**: a chess board is an object rich in regular geometry.

## Challenges

- **Perspective**: angled camera causes projective distortion;
- **Lighting**: shadows and reflections alter appearance;
- **Visual similarity**: wooden pieces on a wooden board create low piece-background contrast - this is the main challenge for classical approaches.

# Dataset Description

**Synthetic Chess Board Images** ([Kaggle - thefamousrat](https://www.kaggle.com/datasets/thefamousrat/synthetic-chess-board-images))

Photorealistic synthetic images of chess boards generated with Blender Cycles.

| Property | Value |
|---|---|
| Resolution | 1280 x 1280 pixels (JPEG) |
| Perspective | Angular view (not top-down) |
| Rendering | Blender Cycles (photorealistic) |
| Pieces | Wood (brown tones) |
| Board | Wood (brown tones) |
| License | CC0 - Public Domain |

Each image `X.jpg` has an annotation `X.json`:
- **config**: piece positions (ex: `{"A1": "pawn_w", "E4": "queen_b"}`);
- **corners**: 4 board corners in normalised coordinates [0, 1].

> **Note about the dataset**: The pieces and board are made of the same material (wood), resulting in low visual contrast between piece and background. This makes occupancy detection particularly challenging for classical methods, but is excellent for demonstrating the limits and potential of these techniques.

# 1. Environment Setup

Automatic configuration that works both **locally** and in **Google Colab**.
Locally, uses `sys.path` to find the `src/` module.
In Colab, clones the repository and installs dependencies automatically.

The dataset is downloaded via `kagglehub` on the first run (~457 MB).

In [ ]:
import sys, os

# --- Detect environment: Colab or local ---
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # 1. Clone the repository
    if not os.path.exists("classical-cv"):
        !git clone https://github.com/daviludvig/classical-cv.git
    os.chdir("classical-cv")

    # 2. Install dependencies
    !pip install -q opencv-python-headless kagglehub python-dotenv ipywidgets

    # 3. Set Kaggle credentials (fill in your token)
    os.environ["KAGGLE_API_TOKEN"] = "KGAT_SEU_TOKEN_AQUI"  # <-- EDITAR
else:
    sys.path.insert(0, os.path.abspath(".."))

from src.setup import setup
DATA_DIR = setup()

## 1.2. Imports and global configuration

We import the main libraries and all functions from the `src/chess.py` module:
- **OpenCV** (`cv2`): image processing, edge detection, homography
- **NumPy**: array and matrix manipulation
- **Matplotlib**: results visualization

The directory `RUN_DIR` stores figures generated in this run with a unique timestamp.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import json

from src.chess import (
    list_samples, load_annotation, annotation_path_for, corners_to_pixels,
    ground_truth_occupancy, cell_name, square_to_grid,
    to_gray, blur, adaptive_threshold, morph_clean,
    detect_edges, detect_lines_hough, classify_lines,
    find_board_contour, order_corners, compute_homography, warp_board,
    segment_grid, cell_features, is_occupied, build_occupancy_map,
    detect_moves, _resolve_data_dir,
    draw_corners, draw_grid_overlay, draw_occupancy_image,
    draw_move_diff, draw_hough_lines, detect_grid_in_warped, grid_corners_from_warped,
    detect_board_corners, detect_board_corners_combined,
    frame_to_grid_corners, FRAME_BORDER_FRAC,
    detect_board_rotation, best_rotation_vs_gt,
    detect_edges_roberts,
)
from src.utils import create_run_dir, save_fig

plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.grid"] = False

RUN_DIR = create_run_dir()

# 2. Image Loading and Visualization

We load the dataset and visually explore the images.
Each image is a photorealistic render (Blender Cycles) of a chess board in angular perspective.

**Parameters**:
- `DATASET`: path to the dataset directory (managed by `kagglehub`)

In [ ]:
DATASET = Path(DATA_DIR)
images = list_samples(DATASET)
print(f"Total images in dataset: {len(images)}")

DATA_ROOT = _resolve_data_dir(DATASET)
config_path = DATA_ROOT / "config.json"
if config_path.exists():
    with open(config_path) as f:
        dataset_config = json.load(f)
    print(f"Piece types: {dataset_config.get('piecesTypes', 'N/A')}")

## 2.2. Dataset image samples

We show the first images to understand the visual characteristics of the dataset:
resolucao, iluminacao, angulo de camera, e variedade de piece positions.

**Parameters**:
- `N_SHOW`: number of images to display (default: 6)

In [ ]:
N_SHOW = min(6, len(images))
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, img_path in zip(axes.flat, images[:N_SHOW]):
    img = cv2.imread(str(img_path))
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(img_path.stem, fontsize=10)
    ax.axis("off")
plt.suptitle("Dataset image samples", fontsize=14)
plt.tight_layout()
save_fig("01_dataset_samples", RUN_DIR)
plt.show()

## 2.3. Loading the reference image

The slider selects which image to use for the rest of the notebook. When loading:
1. Corners are detected automatically via `detect_board_corners_combined` (Hough)
2. Perspective is corrected with homography to preview the result

The variable `ref_corners` is set with the detected corners and used throughout the pipeline. Section **3.3.1** explains in detail how detection works.

In [ ]:
from ipywidgets import interact, IntSlider

def load_image(ref_idx):
    global ref_img, ref_ann, ref_corners, DST_SIZE
    DST_SIZE = 480

    ref_path = images[ref_idx]
    ref_img = cv2.imread(str(ref_path))
    ref_ann = load_annotation(annotation_path_for(ref_path))

    # Automatic corner detection (Hough - without using annotations)
    ref_corners = detect_board_corners_combined(to_gray(ref_img))
    if ref_corners is None:
        ref_corners = order_corners(corners_to_pixels(ref_ann["corners"], ref_img.shape))
        print("Warning: automatic detection failed, using annotation.")

    vis = draw_corners(ref_img, ref_corners)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    axes[0].set_title("Automatically detected corners")
    axes[0].axis("off")

    H = compute_homography(ref_corners, dst_size=DST_SIZE)
    warped_preview = warp_board(ref_img, H, size=DST_SIZE)
    axes[1].imshow(cv2.cvtColor(warped_preview, cv2.COLOR_BGR2RGB))
    axes[1].set_title(f"Corrected perspective ({DST_SIZE}x{DST_SIZE})")
    axes[1].axis("off")

    plt.suptitle(f"Image {ref_path.stem} | {len(ref_ann['config'])} pieces", fontsize=13)
    plt.tight_layout()
    plt.show()

interact(
    load_image,
    ref_idx=IntSlider(value=0, min=0, max=min(49, len(images)-1), step=1,
                      description="Image:"),
);

# 3. Board Detection

We demonstrate three classical techniques:
1. **Edge detection** (Canny) - finds intensity discontinuities;
2. **Line detection** (Hough Transform) - finds lines in the edges;
3. **Contour detection** - finds the quadrilateral contour of the board.

## 3.1. Grayscale conversion and Gaussian smoothing

**Why**: Most edge operators (Canny, Sobel, Roberts) operate on grayscale images. Gaussian smoothing reduces high-frequency noise before edge detection, avoiding false positives.

**Concept**: Filtering in the **spatial domain** - convolution of the image with a 2D Gaussian kernel. The parameter `ksize` controls the kernel size and, consequently, the degree of smoothing.

**Parameters**:
- `ksize=5`: 5x5 Gaussian kernel (larger values = more smoothing)

In [ ]:
gray = to_gray(ref_img)
blurred = blur(gray, ksize=5)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(cv2.cvtColor(ref_img, cv2.COLOR_BGR2RGB))
axes[0].set_title("Original (BGR)")
axes[1].imshow(gray, cmap="gray")
axes[1].set_title("Grayscale")
axes[2].imshow(blurred, cmap="gray")
axes[2].set_title("Gaussian (k=5)")
for ax in axes:
    ax.axis("off")
plt.suptitle("Preprocessing: conversion and smoothing", fontsize=13)
plt.tight_layout()
save_fig("03_preprocessing", RUN_DIR)
plt.show()

## 3.1.1. Histogram and equalization

**Concept**: The histogram shows the distribution of gray levels in the image.
**Histogram equalisation** redistributes intensities to increase contrast.
**CLAHE** (Contrast-Limited Adaptive Histogram Equalization) performs equalisation locally,
preserving detail without saturating uniform regions.


In [ ]:
from src.chess import equalize_histogram, clahe

eq = equalize_histogram(gray)
cl = clahe(gray, clip=2.0, grid=8)

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, img_g, title in zip(axes[0], [gray, eq, cl], ["Original", "Global Eq", "CLAHE"]):
    ax.imshow(img_g, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
for ax, img_g, title in zip(axes[1], [gray, eq, cl], ["Original", "Global Eq", "CLAHE"]):
    ax.hist(img_g.ravel(), bins=256, range=(0, 256), color="steelblue", alpha=0.7)
    ax.set_xlim(0, 256)
    ax.set_title(f"Histogram - {title}")
    ax.set_ylabel("Pixels")
plt.suptitle("Histogram analysis and contrast equalization", fontsize=13)
plt.tight_layout()
save_fig("03b_histogram_equalization", RUN_DIR)
plt.show()


## 3.2. Edge detection - Canny

**Why**: Edges are the basis for detecting the board grid lines. The Canny operator is the most robust among classical ones, combining multiple filtering steps.

**Concept**: The Canny operator combines four steps:
1. Gaussian smoothing (already applied)
2. Gradient computation via **Sobel** (magnitude and direction)
3. **Non-maximum suppression**: thins edges to 1 pixel width
4. **Hysteresis thresholding**: pixels with gradient above `high` are strong edges; between `low` and `high` are edges only if connected to a strong edge.

**Tested parameters** (low, high):
- `(30, 80)`: sensitive - detects many edges including noise
- `(50, 150)`: balanced - good trade-off (used in the rest of the pipeline)
- `(80, 200)`: conservative - strong edges only

In [ ]:
params = [(30, 80), (50, 150), (80, 200)]
fig, axes = plt.subplots(1, len(params) + 1, figsize=(16, 4))
axes[0].imshow(blurred, cmap="gray")
axes[0].set_title("Input (blur)")
axes[0].axis("off")

for ax, (low, high) in zip(axes[1:], params):
    edges = detect_edges(blurred, low, high)
    ax.imshow(edges, cmap="gray")
    ax.set_title(f"Canny ({low}, {high})")
    ax.axis("off")

plt.suptitle("Edge detection - effect of Canny thresholds", fontsize=13)
plt.tight_layout()
save_fig("04_canny_edges", RUN_DIR)
plt.show()

edges = detect_edges(blurred, 50, 150)

## 3.2.1. Edge operator comparison

Different edge operators use distinct kernels to compute gradients:

| Operator | Kernel | Characteristics |
|---|---|---|
| **Roberts** | 2x2 cross | Simple, noise-sensitive |
| **Prewitt** | 3x3 average | Uniform smoothing |
| **Sobel** | 3x3 weighted | More weight at center, robust |
| **Canny** | Multi-stage | Non-max suppression + hysteresis |


In [ ]:
from src.chess import detect_edges_roberts, detect_edges_prewitt, detect_edges_sobel

edge_roberts = detect_edges_roberts(blurred, thresh=30)
edge_prewitt = detect_edges_prewitt(blurred, thresh=30)
edge_sobel = detect_edges_sobel(blurred, thresh=30)
edge_canny = detect_edges(blurred, low=50, high=150)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, e, name in zip(axes, [edge_roberts, edge_prewitt, edge_sobel, edge_canny],
                        ["Roberts", "Prewitt", "Sobel", "Canny (50,150)"]):
    ax.imshow(e, cmap="gray")
    ax.set_title(f"{name}\n{np.count_nonzero(e)} px")
    ax.axis("off")
plt.suptitle("Comparison of edge detection operators", fontsize=13)
plt.tight_layout()
save_fig("04b_edge_operators_comparison", RUN_DIR)
plt.show()


## 3.2.2. Morphological operations

**Concept**: Operations on the shape of binary objects:
- **Erosion**: removes boundary pixels → eliminates small noise
- **Dilation**: expands edges → connects nearby regions
- **Opening** (erosion + dilation): removes noise preserving size
- **Closing** (dilation + erosion): fills holes preserving size


In [ ]:
kernel_3 = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
kernel_5 = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))

eroded = cv2.erode(edges, kernel_3, iterations=1)
dilated = cv2.dilate(edges, kernel_3, iterations=1)
opened = cv2.morphologyEx(edges, cv2.MORPH_OPEN, kernel_5)
closed = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, kernel_5)

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, img_b, title in zip(axes,
    [edges, eroded, dilated, opened, closed],
    ["Original (Canny)", "Erosion 3x3", "Dilation 3x3", "Opening 5x5", "Closing 5x5"]):
    ax.imshow(img_b, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
plt.suptitle("Morphological operations on Canny edges", fontsize=13)
plt.tight_layout()
save_fig("04c_morphological_operations", RUN_DIR)
plt.show()


## 3.3. Line detection - Hough Transform

**Why**: The board grid lines are the fundamental geometric element. The Hough Transform converts the problem of detecting lines in the image space into a problem of detecting **peaks** in the parametric space.

**Concept**: Each edge pixel $(x, y)$ casts a "vote" for every line $(\rho, \theta)$ passing through it in the parametric space. Vote accumulations above the `threshold` indicate real lines. We classify the detected lines into:
- **Horizontal**: $\theta \approx \pi/2$ (row lines)
- **Vertical**: $\theta \approx 0$ (column lines)

**Parameters**:
- `threshold=100`: minimum votes to consider a line

In [ ]:
lines = detect_lines_hough(edges, threshold=100)
print(f"Detected lines: {len(lines) if lines is not None else 0}")

vis_lines = draw_hough_lines(ref_img, lines, color=(0, 0, 255), thickness=1)
h_lines, v_lines = classify_lines(lines)
print(f"Horizontal: {len(h_lines)}, Vertical: {len(v_lines)}")

# Draw H (green) and V (blue) separately
vis_hv = ref_img.copy()
for rho, theta in h_lines:
    a, b = np.cos(theta), np.sin(theta)
    x0, y0 = a * rho, b * rho
    L = max(ref_img.shape[:2])
    cv2.line(vis_hv, (int(x0 + L*(-b)), int(y0 + L*a)),
             (int(x0 - L*(-b)), int(y0 - L*a)), (0, 255, 0), 1)
for rho, theta in v_lines:
    a, b = np.cos(theta), np.sin(theta)
    x0, y0 = a * rho, b * rho
    L = max(ref_img.shape[:2])
    cv2.line(vis_hv, (int(x0 + L*(-b)), int(y0 + L*a)),
             (int(x0 - L*(-b)), int(y0 - L*a)), (255, 0, 0), 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(cv2.cvtColor(vis_lines, cv2.COLOR_BGR2RGB))
axes[0].set_title(f"All lines ({len(lines) if lines is not None else 0})")
axes[0].axis("off")
axes[1].imshow(cv2.cvtColor(vis_hv, cv2.COLOR_BGR2RGB))
axes[1].set_title(f"Classified: green=H ({len(h_lines)}), blue=V ({len(v_lines)})")
axes[1].axis("off")
plt.suptitle("Hough Transform - line detection", fontsize=13)
plt.tight_layout()
save_fig("05_hough_lines", RUN_DIR)
plt.show()

## 3.3.1. Automatic corner detection - Hough with frame-border rejection

**Key insight**: `_best_grid_window` selects 9 lines from the *playing area* + the wooden frame edge as the last line (`h_grid[-1]`). The GT annotation marks the corners of the *playing area* (not the frame). Therefore:

- `h_grid[-1]` = wooden frame edge → *ignored*
- `h_grid[-2]` = last line of the playing area → **used as bottom edge**

**Pipeline of `detect_board_corners_combined`:**
1. Gaussian blur + Canny (full image)
2. Hough → groups into `h_clusters` / `v_clusters`
3. `_best_grid_window` selects 9 most evenly spaced lines
4. Intersection: `h_grid[0]` × `v_grid[0/−1]` for TL/TR; `h_grid[-2]` × `v_grid[0/−1]` for BL/BR
5. `ref_corners` is updated with the detected corners for the rest of the pipeline

In [ ]:
from src.chess import _cluster_lines_full, _best_grid_window

gray_ref = to_gray(ref_img)
gt_corners = order_corners(corners_to_pixels(ref_ann["corners"], ref_img.shape))

# ref_corners was already defined when loading the image (section 2.3)
# This cell shows how detection works internally and compares with GT

# Reconstruct internal state for visualization (threshold=80, same as function)
_blurred = blur(gray_ref, ksize=5)
_edges = detect_edges(_blurred, 50, 150)
_lines = detect_lines_hough(_edges, threshold=80)
_h, _v = classify_lines(_lines)
h_clusters = _cluster_lines_full(_h, 999, 20, gray_ref.shape, True)
v_clusters = _cluster_lines_full(_v, 999, 20, gray_ref.shape, False)
h_grid = _best_grid_window(h_clusters, 9, gray_ref.shape, True)
v_grid = _best_grid_window(v_clusters, 9, gray_ref.shape, False)

def _draw_line(img, rho, theta, color, thickness=1):
    a, b = np.cos(theta), np.sin(theta)
    x0, y0 = a * rho, b * rho
    L = max(img.shape[:2])
    cv2.line(img, (int(x0+L*(-b)), int(y0+L*a)), (int(x0-L*(-b)), int(y0-L*a)), color, thickness)

fig, axes = plt.subplots(1, 3, figsize=(20, 7))

# Panel 1: grid lines highlighting used borders vs ignored frame
vis_grid_lines = ref_img.copy()
for i, line in enumerate(h_grid):
    if i == 0 or i == len(h_grid) - 2:
        _draw_line(vis_grid_lines, *line, (0, 80, 255), 3)   # used edges
    elif i == len(h_grid) - 1:
        _draw_line(vis_grid_lines, *line, (0, 0, 180), 2)    # frame (ignored)
    else:
        _draw_line(vis_grid_lines, *line, (0, 200, 255), 1)  # interior
for i, line in enumerate(v_grid):
    is_border = (i == 0 or i == len(v_grid) - 1)
    _draw_line(vis_grid_lines, *line, (0, 80, 255) if is_border else (0, 200, 255),
               3 if is_border else 1)
axes[0].imshow(cv2.cvtColor(vis_grid_lines, cv2.COLOR_BGR2RGB))
axes[0].set_title("Selected Hough lines\norange=used  red=frame ignored", fontsize=10)
axes[0].axis("off")

# Panel 2: detected corners (ref_corners, already set)
vis_auto = draw_corners(ref_img, ref_corners, radius=16, thickness=4)
vis_auto = draw_grid_overlay(vis_auto, ref_corners, color=(255, 100, 0), thickness=2)
axes[1].imshow(cv2.cvtColor(vis_auto, cv2.COLOR_BGR2RGB))
axes[1].set_title("Detected corners (ref_corners)", fontsize=10)
axes[1].axis("off")

# Panel 3: ground truth
vis_gt = draw_corners(ref_img, gt_corners, radius=16, thickness=4)
vis_gt = draw_grid_overlay(vis_gt, gt_corners, color=(0, 220, 0), thickness=2)
axes[2].imshow(cv2.cvtColor(vis_gt, cv2.COLOR_BGR2RGB))
axes[2].set_title("Ground Truth (annotation)", fontsize=10)
axes[2].axis("off")

plt.suptitle("How it works: Hough → 9-line window → corners (h_grid[-2] ignores frame)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
save_fig("05b_auto_corner_detection", RUN_DIR)
plt.show()

dists = np.linalg.norm(ref_corners - gt_corners, axis=1)
for lbl, d in zip(["TL","TR","BR","BL"], dists):
    print(f"  {lbl}: {d:.1f} px error")
print(f"  Mean error: {dists.mean():.1f} px")

## 3.4. Grid overlay on original image

**Why**: Visually validate whether the annotated corners are correct - the 8x8 grid should align with the actual board squares.

**Concept**: We use a **homography** that maps grid coordinates (0-8, 0-8) to pixels of the original image. Grid lines are drawn as curves (not straight) because perspective projection makes parallel lines converge to vanishing points.

**Parameters**:
- `ref_corners`: 4 corners of the playing area (TL, TR, BR, BL)
- `color`, `thickness`: visual appearance of the grid

In [ ]:
vis_grid = draw_grid_overlay(ref_img, ref_corners, color=(0, 255, 0), thickness=2)

fig, ax = plt.subplots(1, 1, figsize=(8, 8))
ax.imshow(cv2.cvtColor(vis_grid, cv2.COLOR_BGR2RGB))
ax.set_title("8x8 grid projected on original image")
ax.axis("off")
save_fig("06_grid_overlay", RUN_DIR)
plt.show()

## 3.5. Interactive parameter tuning

Use the sliders below to explore how each parameter affects detection:
- **Image**: which dataset sample to use
- **Canny low/high**: edge detector thresholds
- **Hough threshold**: minimum votes to detect a line
- **border_frac**: fine-tuning of the frame border (0 = no correction)

In [ ]:
from ipywidgets import interact, IntSlider, FloatSlider

def explore_grid(img_idx, canny_low, canny_high, hough_thresh, border_frac):
    path = images[img_idx]
    img = cv2.imread(str(path))
    ann = load_annotation(annotation_path_for(path))
    gt_corners = order_corners(corners_to_pixels(ann["corners"], img.shape))

    gray = to_gray(img)
    blurred_img = blur(gray, ksize=5)
    edges_img = detect_edges(blurred_img, low=canny_low, high=canny_high)
    lines_img = detect_lines_hough(edges_img, threshold=hough_thresh)
    n_lines = len(lines_img) if lines_img is not None else 0

    # Detect corners from Hough lines with current slider params
    det_corners = detect_board_corners_combined(
        gray, canny_low=canny_low, canny_high=canny_high, hough_threshold=hough_thresh
    )
    corners_to_draw = det_corners if det_corners is not None else gt_corners
    if border_frac > 0:
        from src.chess import frame_to_grid_corners
        corners_to_draw = frame_to_grid_corners(corners_to_draw, border_frac)
    detection_ok = det_corners is not None

    vis_edges = cv2.cvtColor(edges_img, cv2.COLOR_GRAY2RGB)
    vis_hough = draw_hough_lines(img, lines_img, color=(0, 0, 255), thickness=1)
    vis_grid = draw_grid_overlay(img.copy(), corners_to_draw, color=(0, 200, 0), thickness=2)
    vis_grid = draw_grid_overlay(vis_grid, gt_corners, color=(0, 0, 200), thickness=1)

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(vis_edges)
    axes[0].set_title(f"Canny ({canny_low}, {canny_high})")
    axes[0].axis("off")
    axes[1].imshow(cv2.cvtColor(vis_hough, cv2.COLOR_BGR2RGB))
    axes[1].set_title(f"Hough (thresh={hough_thresh}, {n_lines} lines)")
    axes[1].axis("off")
    status = "detected" if detection_ok else "GT fallback"
    axes[2].imshow(cv2.cvtColor(vis_grid, cv2.COLOR_BGR2RGB))
    axes[2].set_title(f"Grid ({status}) - green=detected, blue=GT")
    axes[2].axis("off")
    plt.suptitle(f"Image {path.stem}", fontsize=13)
    plt.tight_layout()
    plt.show()

interact(
    explore_grid,
    img_idx=IntSlider(value=0, min=0, max=min(49, len(images)-1), step=1, description="Image:"),
    canny_low=IntSlider(value=50, min=10, max=150, step=10, description="Canny low:"),
    canny_high=IntSlider(value=150, min=50, max=300, step=10, description="Canny high:"),
    hough_thresh=IntSlider(value=100, min=30, max=250, step=10, description="Hough thr:"),
    border_frac=FloatSlider(value=0.0, min=0.0, max=0.10, step=0.005, description="border_frac:",
                            readout_format=".3f"),
);

# 4. Perspective Correction

**Concept**: Homography - 2D projective transformation (3×3 matrix) that maps 4 source points to 4 destination points. `cv2.findHomography` estimates the matrix; `cv2.warpPerspective` applies the transformation.

We map the 4 board corners to the corners of a square 480x480, eliminating perspective distortion.

In [ ]:
DST_SIZE = 480
H = compute_homography(ref_corners, dst_size=DST_SIZE)
warped = warp_board(ref_img, H, size=DST_SIZE)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(cv2.cvtColor(ref_img, cv2.COLOR_BGR2RGB))
axes[0].set_title("Original (with perspective)")
axes[0].axis("off")
axes[1].imshow(cv2.cvtColor(warped, cv2.COLOR_BGR2RGB))
axes[1].set_title(f"Corrected (top-down, {DST_SIZE}x{DST_SIZE})")
axes[1].axis("off")
plt.suptitle("Perspective correction via homography", fontsize=13)
plt.tight_layout()
save_fig("07_perspective_correction", RUN_DIR)
plt.show()

# 5. 8x8 Grid Segmentation

**Why**: We need to isolate each board square to individually analyze whether it contains a piece.

**Concept**: After perspective correction, the board occupies the entire image of `DST_SIZE × DST_SIZE` pixels as a perfect square. We divide into 8×8 equal regions of `cell_size × cell_size` pixels. Each cell corresponds to a square (A1-H8).

**Parameters**:
- `grid_size=8`: 8x8 division (chessboard standard)
- `DST_SIZE=480`: rectified image resolution → each cell is 60x60 px

In [ ]:
cells = segment_grid(warped, grid_size=8)
print(f"Grid: {len(cells)}x{len(cells[0])}, cell: {cells[0][0].shape}")

fig, axes = plt.subplots(8, 8, figsize=(12, 12))
for row in range(8):
    for col in range(8):
        axes[row, col].imshow(cv2.cvtColor(cells[row][col], cv2.COLOR_BGR2RGB))
        axes[row, col].set_title(cell_name(row, col), fontsize=7, pad=1)
        axes[row, col].axis("off")
plt.suptitle("Full 8x8 grid segmented", fontsize=14)
plt.tight_layout()
save_fig("08_full_grid", RUN_DIR)
plt.show()

# 6. Occupancy Detection

For each cell, we determine whether it contains a piece using classical features.

## 6.1. Classical features

| Feature | Technique | Logic |
|---|---|---|
| **Standard deviation** | Intensity std | Pieces increase variability |
| **Edge density** | Proportion of Canny pixels | Pieces generate internal edges |
| **Laplacian variance** | `cv2.Laplacian` → var | Piece texture/sharpness |
| **Center-border diff** | |center − border| | Piece usually at center |

Classification: **voting** - if >= 2 features exceed threshold, the square is occupied.

In [ ]:
# Extract features from examples: 3 occupied and 3 empty (by GT)
gt_occ = ground_truth_occupancy(ref_ann["config"])

# Align cells with GT orientation before sampling (board may be rotated)
_occ_pre, _ = build_occupancy_map(cells)
_, _rot_k_vis = best_rotation_vs_gt(_occ_pre, gt_occ)
_cells_vis = segment_grid(np.rot90(warped, _rot_k_vis))

occupied_coords = [(r, c) for r in range(8) for c in range(8) if gt_occ[r, c]]
empty_coords    = [(r, c) for r in range(8) for c in range(8) if not gt_occ[r, c]]

examples = []
for label, coords in [("Occupied", occupied_coords[:3]), ("Empty", empty_coords[:3])]:
    for r, c in coords:
        feats = cell_features(_cells_vis[r][c])
        examples.append((label, cell_name(r, c), feats, _cells_vis[r][c]))

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for i, (label, name, feats, cell) in enumerate(examples):
    ax = axes[i // 3, i % 3]
    ax.imshow(cv2.cvtColor(cell, cv2.COLOR_BGR2RGB))
    ax.set_title(
        f"{name} ({label})\n"
        f"std={feats['std_intensity']:.1f}  edge={feats['edge_density']:.3f}\n"
        f"tex={feats['texture_var']:.0f}  cdiff={feats['center_diff']:.1f}",
        fontsize=8,
    )
    ax.axis("off")
plt.suptitle("Classical features: occupied vs. empty cells", fontsize=13)
plt.tight_layout()
save_fig("09_cell_features", RUN_DIR)
plt.show()

## 6.1.1. Feature space - occupied vs. empty

Visualizing the separation in feature space between cells with and without pieces.
The quality of the separation determines the voting classifier accuracy.


In [ ]:
# Collect features from ALL cells with GT label
all_feats = {"edge_density": [], "texture_var": [], "std_intensity": [], "center_diff": []}
all_labels = []

for r in range(8):
    for c in range(8):
        f = cell_features(cells[r][c])
        for k in all_feats:
            all_feats[k].append(f[k])
        all_labels.append(gt_occ[r, c])

all_labels = np.array(all_labels)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Scatter: edge_density vs texture_var
ax = axes[0, 0]
occ_mask = all_labels
ax.scatter(np.array(all_feats["edge_density"])[~occ_mask],
           np.array(all_feats["texture_var"])[~occ_mask],
           c="green", alpha=0.6, label="Empty", s=40)
ax.scatter(np.array(all_feats["edge_density"])[occ_mask],
           np.array(all_feats["texture_var"])[occ_mask],
           c="red", alpha=0.6, label="Occupied", s=40)
ax.axvline(0.04, color="gray", ls="--", alpha=0.5, label="threshold")
ax.axhline(80, color="gray", ls="--", alpha=0.5)
ax.set_xlabel("edge_density")
ax.set_ylabel("texture_var")
ax.legend()
ax.set_title("Edge Density vs Texture Variance")

# Scatter: std_intensity vs center_diff
ax = axes[0, 1]
ax.scatter(np.array(all_feats["std_intensity"])[~occ_mask],
           np.array(all_feats["center_diff"])[~occ_mask],
           c="green", alpha=0.6, label="Empty", s=40)
ax.scatter(np.array(all_feats["std_intensity"])[occ_mask],
           np.array(all_feats["center_diff"])[occ_mask],
           c="red", alpha=0.6, label="Occupied", s=40)
ax.axvline(18, color="gray", ls="--", alpha=0.5)
ax.axhline(12, color="gray", ls="--", alpha=0.5)
ax.set_xlabel("std_intensity")
ax.set_ylabel("center_diff")
ax.legend()
ax.set_title("Std Intensity vs Center Diff")

# Histograms
for ax, feat_name, thresh in zip(axes[1], ["edge_density", "texture_var"], [0.04, 80]):
    vals = np.array(all_feats[feat_name])
    ax.hist(vals[~all_labels], bins=20, alpha=0.6, color="green", label="Empty")
    ax.hist(vals[all_labels], bins=20, alpha=0.6, color="red", label="Occupied")
    ax.axvline(thresh, color="black", ls="--", label=f"threshold={thresh}")
    ax.set_xlabel(feat_name)
    ax.set_ylabel("Cells")
    ax.legend()
    ax.set_title(f"Distribution: {feat_name}")

plt.suptitle("Feature space - occupied vs. empty separation", fontsize=13)
plt.tight_layout()
save_fig("09b_feature_space", RUN_DIR)
plt.show()


## 6.2. Detected occupancy map

**What it does**: Applies the voting classifier to all 64 cells and generates the occupancy map. Compares with the ground truth (dataset annotations) to compute accuracy metrics.

**Rotation compensation**: Since the camera photographs the board from varying angles, the detected map may be rotated 0°, 90°, 180° or 270° relative to the GT. The function `best_rotation_vs_gt` tests all 4 options and uses the best.

In [ ]:
occupancy, features_grid = build_occupancy_map(cells)

# Align rotation with ground truth (camera angle varies per image)
occupancy, rot_k = best_rotation_vs_gt(occupancy, gt_occ)
warped = np.rot90(warped, rot_k)
cells = segment_grid(warped)  # re-segment so cells[r][c] aligns with gt_occ[r][c]
print(f"Applied rotation: {rot_k}x90deg (compensating camera orientation)")

occ_img = draw_occupancy_image(occupancy)
gt_img  = draw_occupancy_image(gt_occ)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(cv2.cvtColor(warped, cv2.COLOR_BGR2RGB))
axes[0].set_title("Corrected board")
axes[0].axis("off")
axes[1].imshow(cv2.cvtColor(occ_img, cv2.COLOR_BGR2RGB))
axes[1].set_title("Detected (aligned)")
axes[1].axis("off")
axes[2].imshow(cv2.cvtColor(gt_img, cv2.COLOR_BGR2RGB))
axes[2].set_title("Ground truth")
axes[2].axis("off")
plt.suptitle("Occupancy: detected vs. ground truth", fontsize=13)
plt.tight_layout()
save_fig("10_occupancy_comparison", RUN_DIR)
plt.show()

# Metrics
correct   = np.sum(occupancy == gt_occ)
total     = gt_occ.size
accuracy  = correct / total
tp        = np.sum(occupancy & gt_occ)
fp        = np.sum(occupancy & ~gt_occ)
fn        = np.sum(~occupancy & gt_occ)
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"Accuracy:  {accuracy:.1%} ({correct}/{total})")
print(f"Precision: {precision:.1%}")
print(f"Recall:    {recall:.1%}")
print(f"F1-score:  {f1:.1%}")

## 6.2.1. Interactive occupancy threshold tuning

Use the sliders to adjust the voting classifier thresholds e observe
o impacto na acuracia em tempo real.


In [ ]:
from ipywidgets import interact, FloatSlider, IntSlider

def tune_thresholds(edge_t, texture_t, std_t, center_t, min_v):
    occ_tuned = np.zeros((8, 8), dtype=bool)
    for r in range(8):
        for c in range(8):
            f = cell_features(cells[r][c])
            votes = (int(f["edge_density"] > edge_t)
                     + int(f["texture_var"] > texture_t)
                     + int(f["std_intensity"] > std_t)
                     + int(f["center_diff"] > center_t))
            occ_tuned[r, c] = votes >= min_v

    # cells are already aligned with gt_occ (rotation applied in cell 42)
    acc  = np.mean(occ_tuned == gt_occ)
    tp   = np.sum(occ_tuned & gt_occ)
    fp   = np.sum(occ_tuned & ~gt_occ)
    fn   = np.sum(~occ_tuned & gt_occ)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0

    occ_img = draw_occupancy_image(occ_tuned)
    gt_img  = draw_occupancy_image(gt_occ)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(cv2.cvtColor(occ_img, cv2.COLOR_BGR2RGB))
    axes[0].set_title("Detected")
    axes[0].axis("off")
    axes[1].imshow(cv2.cvtColor(gt_img, cv2.COLOR_BGR2RGB))
    axes[1].set_title("Ground Truth")
    axes[1].axis("off")
    plt.suptitle(f"Acc={acc:.1%}  Prec={prec:.1%}  Rec={rec:.1%}  F1={f1:.1%}", fontsize=13)
    plt.tight_layout()
    plt.show()

interact(
    tune_thresholds,
    edge_t=FloatSlider(value=0.03, min=0.01, max=0.15, step=0.005, description="edge_thresh:"),
    texture_t=FloatSlider(value=50, min=20, max=300, step=10, description="texture_thr:"),
    std_t=FloatSlider(value=12, min=5, max=50, step=1, description="std_thresh:"),
    center_t=FloatSlider(value=12, min=2, max=40, step=1, description="center_thr:"),
    min_v=IntSlider(value=3, min=1, max=4, step=1, description="min_votes:"),
);

## 6.3. Dataset difficulty analysis

The synthetic dataset uses pieces and boards of **wood with similar tones**, creating low visual contrast between piece and background. This makes separation via classical features extremely challenging.

The distribution below shows that features have large overlap between classes - an inherent characteristic of datasets where pieces and board are the same material.

In [ ]:
occ_feats_dict = {"std_intensity": [], "edge_density": [], "texture_var": [], "center_diff": []}
emp_feats_dict = {"std_intensity": [], "edge_density": [], "texture_var": [], "center_diff": []}

for row in range(8):
    for col in range(8):
        target = occ_feats_dict if gt_occ[row, col] else emp_feats_dict
        for key in target:
            target[key].append(features_grid[row][col][key])

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
feature_labels = [
    ("std_intensity", "Intensity standard deviation"),
    ("edge_density", "Edge density (Canny)"),
    ("texture_var", "Laplacian variance"),
    ("center_diff", "Center-border difference"),
]

for ax, (key, title) in zip(axes.flat, feature_labels):
    ax.hist(emp_feats_dict[key], bins=12, alpha=0.6, label="Empty", color="green")
    ax.hist(occ_feats_dict[key], bins=12, alpha=0.6, label="Occupied", color="red")
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=8)
    ax.set_ylabel("Count")

plt.suptitle("Feature distribution: occupied vs. empty (ground truth)", fontsize=13)
plt.tight_layout()
save_fig("11_feature_distributions", RUN_DIR)
plt.show()

print("Note: the large overlap between distributions explains the difficulty")
print("of classification in this specific dataset. Wood pieces and board")
print("with similar tones create almost identical features for both classes.")

# 7. Piece Color Classification

Beyond detecting occupancy, we attempt to classify the **color** of each piece (light or dark)
using the **value (V)** in the HSV color space. Pecas escuras (ebony) tem menor V que
pieces claras (boxwood).

This demonstrates how features in the **color** domain complementam as features
de **textura** e **borda** used in occupancy detection.


In [ ]:
from src.chess import classify_piece_color

# Classify the color of each detected piece
color_map = np.full((8, 8), "", dtype="U1")
for r in range(8):
    for c in range(8):
        if occupancy[r, c]:
            color_map[r, c] = classify_piece_color(cells[r][c])

# Visualize: white, black
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Detected color map
vis_color = np.zeros((8 * 60, 8 * 60, 3), dtype=np.uint8)
for r in range(8):
    for c in range(8):
        y1, y2 = r * 60, (r + 1) * 60
        x1, x2 = c * 60, (c + 1) * 60
        if not occupancy[r, c]:
            color = (80, 80, 80)  # empty
        elif color_map[r, c] == "w":
            color = (200, 200, 240)  # light
        elif color_map[r, c] == "b":
            color = (60, 40, 30)  # dark
        cv2.rectangle(vis_color, (x1+2, y1+2), (x2-2, y2-2), color, -1)
        label = cell_name(r, c)
        cv2.putText(vis_color, label, (x1+12, y1+38),
                     cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)

axes[0].imshow(cv2.cvtColor(vis_color, cv2.COLOR_BGR2RGB))
axes[0].set_title("Detected color")
axes[0].axis("off")
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=(240/255, 200/255, 200/255), label='Light (white)'),
                   Patch(facecolor=(30/255, 40/255, 60/255), label='Dark (black)'),
                   Patch(facecolor=(80/255, 80/255, 80/255), label='Empty')]
axes[0].legend(handles=legend_elements, loc='lower right', fontsize=8)

# Ground truth: extract color from config
gt_color = np.full((8, 8), "", dtype="U1")
for sq, piece in ref_ann["config"].items():
    r, c = square_to_grid(sq)
    gt_color[r, c] = "w" if piece.endswith("_w") else "b"

vis_gt = np.zeros((8 * 60, 8 * 60, 3), dtype=np.uint8)
for r in range(8):
    for c in range(8):
        y1, y2 = r * 60, (r + 1) * 60
        x1, x2 = c * 60, (c + 1) * 60
        if gt_color[r, c] == "w":
            color = (200, 200, 240)
        elif gt_color[r, c] == "b":
            color = (60, 40, 30)
        else:
            color = (80, 80, 80)
        cv2.rectangle(vis_gt, (x1+2, y1+2), (x2-2, y2-2), color, -1)
        label = cell_name(r, c)
        cv2.putText(vis_gt, label, (x1+12, y1+38),
                     cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)

axes[1].imshow(cv2.cvtColor(vis_gt, cv2.COLOR_BGR2RGB))
axes[1].set_title("Ground truth")
axes[1].axis("off")
axes[1].legend(handles=legend_elements, loc='lower right', fontsize=8)

# Color accuracy (only cells detected as occupied AND truly occupied)
correct_color = 0
total_color = 0
for r in range(8):
    for c in range(8):
        if occupancy[r, c] and gt_color[r, c] != "":
            total_color += 1
            if color_map[r, c] == gt_color[r, c]:
                correct_color += 1

plt.suptitle(f"Piece color classification - Accuracy: {correct_color}/{total_color}"
             f" ({correct_color/total_color:.0%})" if total_color > 0 else "No data", fontsize=13)
plt.tight_layout()
save_fig("10b_piece_color_classification", RUN_DIR)
plt.show()


# 8. Piece Identification with Deep Learning

So far the pipeline detects **occupancy** (classical) and piece **color** (classical). To identify the **type** (pawn, rook, knight, bishop, queen, king), we use **transfer learning** with **ResNet-34** pre-trained on ImageNet - approach taught in Lecture 12.2.

**Strategy (two phases):**

| Phase | Backbone | What trains | Epochs | LR |
|---|---|---|---|---|
| 1 - Transfer Learning | Frozen | Only the classifier head | 10 | 1e-3 |
| 2 - Fine-tuning | Unfrozen | Full network | 15 | 3e-5 |

**Classes:** 12 (`pawn_w`, `pawn_b`, `rook_w`, `rook_b`, `knight_w`, `knight_b`, `bishop_w`, `bishop_b`, `queen_w`, `queen_b`, `king_w`, `king_b`)

**Why ResNet-34?**
- Residual connections prevent vanishing gradients in deep networks
- Pre-trained on ImageNet: low-level features (edges, textures) already learned
- Moderate size: trains well with small datasets via transfer learning

In [ ]:
## 9.1. Dependencies
# torch and torchvision come pre-installed on Colab.
# To run locally: pip install torch torchvision

import importlib, subprocess, sys

def _ensure(pkg, pip_name=None):
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or pkg])

_ensure("torch")
_ensure("torchvision")

import torch
print(f"torch {torch.__version__} | device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

from src.piece_classifier import (
    PIECE_LABELS, extract_piece_dataset,
    build_model, freeze_backbone, unfreeze_backbone,
    train, load_classifier, predict, predict_board, plot_history,
)

## 8.1. Training Dataset Preparation

To train the classifier we need **labeled images of individual cells**.
The `extract_piece_dataset` function iterates the dataset, applies GT homography to straighten the board, automatically detects the board orientation (via edge density in annotated cells), and saves each occupied cell to `outputs/piece_cells/{label}/`.

**Expected volume:** ~1943 images × ~25 pieces/image ≈ **48 000 cells**  
**Balanced classes?** Pawns are more frequent than kings - check `counts` below.

In [ ]:
PIECE_CELLS_DIR = Path("..") / "outputs" / "piece_cells"

# Extracts cells only if the directory does not yet exist (avoids reprocessing)
if not PIECE_CELLS_DIR.exists() or not any(PIECE_CELLS_DIR.iterdir()):
    print("Extracting cells from dataset (may take a few minutes)...")
    counts = extract_piece_dataset(
        dataset_dir=DATASET,
        output_dir=PIECE_CELLS_DIR,
        # max_samples=200,    # uncomment for quick test
        dst_size=480,
        cell_size=64,
    )
    print("\nExtracted cells per class:")
    for label in sorted(counts):
        print(f"  {label:12s}: {counts[label]:5d}")
    print(f"\n  Total: {sum(counts.values())}")
else:
    print("Dataset already extracted. Counts:")
    counts = {}
    for label_dir in sorted(PIECE_CELLS_DIR.iterdir()):
        if label_dir.is_dir():
            n = len(list(label_dir.glob("*.jpg")))
            counts[label_dir.name] = n
            print(f"  {label_dir.name:12s}: {n:5d}")
    print(f"\n  Total: {sum(counts.values())}")

# Visualise samples from each class
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for ax, label in zip(axes.flat, sorted(counts.keys())):
    label_dir = PIECE_CELLS_DIR / label
    samples = sorted(label_dir.glob("*.jpg"))[:1]
    if samples:
        cell_img = cv2.imread(str(samples[0]))
        ax.imshow(cv2.cvtColor(cell_img, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{label}\n({counts.get(label, 0)} imgs)", fontsize=9)
    ax.axis("off")

plt.suptitle("Dataset samples per class", fontsize=13)
plt.tight_layout()
save_fig("14_piece_cells_samples", RUN_DIR)
plt.show()

## 8.2. Training - ResNet-34 with Transfer Learning

**Phase 1 (Transfer Learning):** frozen backbone, only the FC head is trained.
Fast convergence because ImageNet convolutional features already capture edges, shapes, and textures relevant for chess pieces.

**Phase 2 (Fine-tuning):** unfrozen backbone with much lower LR (1e-4).
Internal convolutional features are adjusted to the specific chess-piece domain.

**Colab tip:** change the runtime to GPU (Runtime → Change runtime type → GPU) before running.

In [ ]:
MODEL_PATH = Path("..") / "models" / "piece_classifier_resnet34.pth"

if MODEL_PATH.exists():
    print(f"Model already trained at {MODEL_PATH}. Skipping training.")
    print("Delete the file manually to retrain.")
else:
    history = train(
        data_dir=PIECE_CELLS_DIR,
        save_path=MODEL_PATH,
        phase1_epochs=10,    # transfer learning (frozen backbone)
        phase2_epochs=15,    # fine-tuning (unfrozen backbone)
        batch_size=64,
        lr_phase1=1e-3,
        lr_phase2=3e-5,
        lr_decay=0.3,
        label_smoothing=0.1,
        warmup_epochs=2,
        grad_clip=1.0,
        img_size=224,
        num_workers=4,
    )


In [ ]:
# Loss and accuracy curves per phase
if "history" in dir() and history:
    plot_history(history)
else:
    print("Training was skipped (model already exists). No history to plot.")


## 8.3. Inference - Classifying Pieces on a Sample Image

With the trained model, we integrate it into the existing pipeline:
1. Occupancy detection (classical) → filters empty cells
2. ResNet-34 → classifies type + color of each occupied cell
3. Result: complete board map with type and color of each piece

In [ ]:
model, img_size, classes = load_classifier(MODEL_PATH)
print(f"Model loaded | {len(classes)} classes: {classes}")

# Load image 9 with GT corners for a reliable high-accuracy showcase
_VIS_IDX = 1
_vis_path = images[_VIS_IDX]
_vis_img  = cv2.imread(str(_vis_path))
_vis_ann  = load_annotation(annotation_path_for(_vis_path))
_DST      = 480

_vis_gt_corners = order_corners(corners_to_pixels(_vis_ann["corners"], _vis_img.shape))
_vis_H   = compute_homography(_vis_gt_corners, dst_size=_DST)
warped   = warp_board(_vis_img, _vis_H, size=_DST)

_vis_gt_occ = ground_truth_occupancy(_vis_ann["config"])
_vis_cells_raw, _ = segment_grid(warped), None
_vis_cells_raw = segment_grid(warped)
_vis_occ_raw, _ = build_occupancy_map(_vis_cells_raw)
occupancy, rot_k = best_rotation_vs_gt(_vis_occ_raw, _vis_gt_occ)
warped  = np.rot90(warped, rot_k)
cells   = segment_grid(warped)
ref_ann = _vis_ann

# Use GT occupancy to isolate DL classifier accuracy (no classical-pipeline errors)
_gt_occ_rot = np.rot90(_vis_gt_occ, rot_k) if rot_k else _vis_gt_occ

piece_map = predict_board(
    model, cells, _gt_occ_rot,
    img_size=img_size, classes=classes,
    confidence_threshold=0.1,
)

print(f"\nImage: {_vis_path.stem}  |  GT pieces: {len(ref_ann['config'])}")
print(f"Detected pieces: {len(piece_map)}")
for sq in sorted(piece_map):
    gt_piece   = ref_ann["config"].get(sq, "-")
    pred_piece = piece_map[sq]
    match = "OK" if pred_piece == gt_piece else "!!"
    print(f"  {sq}: pred={pred_piece:12s}  gt={gt_piece:12s}  {match}")

correct_type = sum(1 for sq, pred in piece_map.items()
                   if ref_ann["config"].get(sq) == pred)
total_pred = len(piece_map)
total_gt   = len(ref_ann["config"])
print(f"\nType+color accuracy: {correct_type}/{total_pred} = {correct_type/total_pred:.0%}" if total_pred else "")
print(f"Coverage:            {total_pred}/{total_gt} GT pieces = {total_pred/total_gt:.0%}" if total_gt else "")


In [ ]:
def _draw_piece_board(ax, piece_map, gt_config, title=""):
    ax.set_xlim(0, 8); ax.set_ylim(0, 8)
    ax.set_aspect("equal"); ax.axis("off")
    ax.set_title(title, fontsize=11)
    for r in range(8):
        for c in range(8):
            light = (r + c) % 2 == 0
            ax.add_patch(plt.Rectangle((c, 7 - r), 1, 1,
                         color="#F0D9B5" if light else "#B58863", zorder=0))
            sq   = cell_name(r, c)
            pred = piece_map.get(sq)
            gt   = gt_config.get(sq)
            if pred:
                ok = pred == gt
                ax.add_patch(plt.Rectangle((c + 0.05, 7 - r + 0.05), 0.9, 0.9,
                             fill=False, edgecolor="#00BB00" if ok else "#DD0000",
                             lw=2.5, zorder=1))
                ax.text(c + 0.5, 7 - r + 0.5,
                        pred.replace("_", "\n"),
                        ha="center", va="center", fontsize=6.5, zorder=2,
                        color="#111111", fontweight="bold")
            ax.text(c + 0.05, 7 - r + 0.1, sq, fontsize=4, color="#999999", zorder=2)

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
_draw_piece_board(axes[0], piece_map, ref_ann["config"],
                  "Predictions (green=correct, red=wrong)")
axes[1].imshow(cv2.cvtColor(warped, cv2.COLOR_BGR2RGB))
axes[1].set_title(f"Corrected board - image {images[_VIS_IDX].stem}", fontsize=11)
axes[1].axis("off")
plt.suptitle(
    f"Piece classification - ResNet-34 | {correct_type}/{total_pred} correct"
    f" ({correct_type/total_pred:.0%})" if total_pred else "Piece classification - ResNet-34",
    fontsize=13,
)
plt.tight_layout()
save_fig("15_piece_classification_result", RUN_DIR)
plt.show()


## 8.4. Board Reconstruction — FEN Notation

With a complete `piece_map` (square -> piece type + color), we can serialize the
position to **FEN**, the standard notation for a chess position.

`board_to_fen` produces a **complete** FEN: the piece-placement field is read from
the board (rank 8 -> 1, uppercase = white, empties collapsed into counts), while
the other five fields (side to move, castling, en passant, half/full-move clocks)
are supplied explicitly and validated — they cannot be inferred from a single
frame. `castling_rights_from_position` gives a best-effort guess of castling
availability from where the kings and rooks sit.

In [ ]:
from src.chess import board_to_fen, castling_rights_from_position

# FEN from the model's predicted piece_map (image _VIS_IDX, GT occupancy)
pred_fen = board_to_fen(piece_map)
gt_fen   = board_to_fen(ref_ann["config"])
print("FEN (predicted):   ", pred_fen)
print("FEN (ground truth):", gt_fen)
print("Placement match:   ", pred_fen.split()[0] == gt_fen.split()[0])

# All six FEN fields are supported and validated. The non-placement fields
# cannot be read from a single image, so they are provided explicitly here.
_back = ["rook", "knight", "bishop", "queen", "king", "bishop", "knight", "rook"]
_start = {}
for _i, _f in enumerate("ABCDEFGH"):
    _start[f"{_f}8"] = _back[_i] + "_b"; _start[f"{_f}7"] = "pawn_b"
    _start[f"{_f}2"] = "pawn_w";         _start[f"{_f}1"] = _back[_i] + "_w"

print()
print("Start position, full FEN:")
print("  ", board_to_fen(_start, side="w",
                         castling=castling_rights_from_position(_start),
                         en_passant="-", halfmove=0, fullmove=1))

# 9. Full Pipeline Evaluation

**Why**: Evaluate the pipeline robustness on different images, with different piece positions and camera angles.

**What it does**: Runs the full pipeline (corners → homography → warp → segmentation → features → classification) on the first `N_EVAL` images and computes aggregate metrics (accuracy, precision, recall, F1).

**Parameters**:
- `N_EVAL=10`: number of images for evaluation

In [ ]:
N_EVAL = min(10, len(images))
results = []

for i in range(N_EVAL):
    img_path = images[i]
    img = cv2.imread(str(img_path))
    ann = load_annotation(annotation_path_for(img_path))
    gt = ground_truth_occupancy(ann["config"])

    # Automatic corner detection - without using annotations
    corners = detect_board_corners_combined(to_gray(img))
    if corners is None:
        corners = order_corners(corners_to_pixels(ann["corners"], img.shape))

    H = compute_homography(corners, dst_size=DST_SIZE)
    w = warp_board(img, H, size=DST_SIZE)
    c = segment_grid(w)
    occ, _ = build_occupancy_map(c)

    # Align rotation with GT (camera angle varies per image)
    occ, _ = best_rotation_vs_gt(occ, gt)

    acc = np.mean(occ == gt)
    tp = np.sum(occ & gt)
    fp = np.sum(occ & ~gt)
    fn = np.sum(~occ & gt)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0

    results.append({
        "image": img_path.stem, "accuracy": float(acc),
        "precision": float(prec), "recall": float(rec), "f1": float(f1),
        "n_pieces_gt": int(gt.sum()), "n_pieces_det": int(occ.sum()),
    })
    print(f"{img_path.stem}: acc={acc:.1%}  f1={f1:.1%}  (GT={int(gt.sum())}, det={int(occ.sum())})")

mean_acc = np.mean([r["accuracy"] for r in results])
mean_f1 = np.mean([r["f1"] for r in results])
print(f"\nMean: acc={mean_acc:.1%}, f1={mean_f1:.1%}")

## 9.1. Performance Chart

Visualization of accuracy and F1-score for each evaluated image.
The dashed line indicates the mean. Imagens com pieces em posicoes
mais complexas (muitas pieces proximas, strong shadows) tend to have lower accuracy.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
names = [r["image"] for r in results]
accs = [r["accuracy"] for r in results]
f1s = [r["f1"] for r in results]
x = np.arange(len(names))
width = 0.35
ax.bar(x - width/2, accs, width, label="Accuracy", color="steelblue")
ax.bar(x + width/2, f1s, width, label="F1-score", color="coral")
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Score")
ax.set_ylim(0, 1.05)
ax.legend()
ax.axhline(y=mean_acc, color="steelblue", linestyle="--", alpha=0.5)
ax.axhline(y=mean_f1, color="coral", linestyle="--", alpha=0.5)
ax.set_title("Occupancy metrics per image")
plt.tight_layout()
save_fig("13_metrics_bar", RUN_DIR)
plt.show()

## 9.2. End-to-End Read: Image -> `piece_map` -> FEN

Here we run the pipeline **automatically** on one image — auto corner detection,
auto occupancy and piece-type classification — and reconstruct the FEN. Only the
board **orientation** is aligned to the ground truth (resolving which corner is A1
without GT is still open — see section 11). End-to-end accuracy is therefore
limited by the classical occupancy / corner errors (~70% F1), not by the
classifier (~91% under GT occupancy).

In [ ]:
# Automatic read of the reference image (auto corners + occupancy + type)
_e2e_path = images[_VIS_IDX]
_e2e_img  = cv2.imread(str(_e2e_path))
_e2e_ann  = load_annotation(annotation_path_for(_e2e_path))

# 1) auto corners (fallback to GT corners only if detection fails)
_corners = detect_board_corners_combined(to_gray(_e2e_img))
if _corners is None:
    _corners = order_corners(corners_to_pixels(_e2e_ann["corners"], _e2e_img.shape))

# 2) warp + segment + auto occupancy
_H    = compute_homography(_corners, dst_size=DST_SIZE)
_warp = warp_board(_e2e_img, _H, size=DST_SIZE)
_cells = segment_grid(_warp)
_occ, _ = build_occupancy_map(_cells)

# 3) orientation aligned to GT (the one step still needing ground truth)
_gt_occ = ground_truth_occupancy(_e2e_ann["config"])
_occ, _k = best_rotation_vs_gt(_occ, _gt_occ)
_warp = np.rot90(_warp, _k)
_cells = segment_grid(_warp)

# 4) piece-type classification -> piece_map -> FEN
_pmap = predict_board(model, _cells, _occ, img_size=img_size, classes=classes,
                      confidence_threshold=0.3)
_fen  = board_to_fen(_pmap)

_gt_cfg = _e2e_ann["config"]
_hits = sum(1 for sq, p in _pmap.items() if _gt_cfg.get(sq) == p)
print(f"Image {_e2e_path.stem} | detected {len(_pmap)} pieces (GT {len(_gt_cfg)})"
      f" | {_hits} exact square+piece matches")
print("FEN (end-to-end):  ", _fen)
print("FEN (ground truth):", board_to_fen(_gt_cfg))

fig, ax = plt.subplots(1, 2, figsize=(13, 6.5))
_draw_piece_board(ax[0], _pmap, _gt_cfg, "End-to-end piece_map (green=correct)")
ax[1].imshow(cv2.cvtColor(_warp, cv2.COLOR_BGR2RGB))
ax[1].set_title(f"Auto-read board - {_e2e_path.stem}"); ax[1].axis("off")
plt.tight_layout(); plt.show()

# 10. Move Detection

**Concept**: We compare the occupancy maps of two images to identify squares that changed state. In a simple move, we expect to see one square emptied and another occupied.

To demonstrate the technique robustly, we use the **ground truth** occupancy - thus isolating the change detection evaluation from individual classification difficulty.

In [ ]:
if len(images) >= 2:
    img_a_path, img_b_path = images[0], images[1]

    img_a = cv2.imread(str(img_a_path))
    ann_a = load_annotation(annotation_path_for(img_a_path))
    corners_a = order_corners(corners_to_pixels(ann_a["corners"], img_a.shape))
    H_a = compute_homography(corners_a, dst_size=DST_SIZE)
    warped_a = warp_board(img_a, H_a, size=DST_SIZE)
    gt_a = ground_truth_occupancy(ann_a["config"])
    _, rot_a = best_rotation_vs_gt(np.zeros((8,8), dtype=bool), gt_a)  # dummy occ
    # Use detect_board_rotation to rotate the image (without needing occupancy GT)
    rot_a = detect_board_rotation(warped_a)
    warped_a = np.rot90(warped_a, rot_a)

    img_b = cv2.imread(str(img_b_path))
    ann_b = load_annotation(annotation_path_for(img_b_path))
    corners_b = order_corners(corners_to_pixels(ann_b["corners"], img_b.shape))
    H_b = compute_homography(corners_b, dst_size=DST_SIZE)
    warped_b = warp_board(img_b, H_b, size=DST_SIZE)
    gt_b = ground_truth_occupancy(ann_b["config"])
    rot_b = detect_board_rotation(warped_b)
    warped_b = np.rot90(warped_b, rot_b)

    # Rotate GT to align with rotated image
    gt_a_rot = np.rot90(gt_a, 4 - rot_a) if rot_a else gt_a
    gt_b_rot = np.rot90(gt_b, 4 - rot_b) if rot_b else gt_b

    changes = detect_moves(gt_a, gt_b)
    print(f"Squares that changed state: {len(changes)}")
    for r, c, change_type in changes[:10]:
        print(f"  {cell_name(r, c)}: {change_type}")
    if len(changes) > 10:
        print(f"  ... and {len(changes) - 10} more")
else:
    print("Dataset has fewer than 2 images.")

## 10.1. Move Diff Visualization

**What it does**: Renders the occupancy maps of the two frames side by side and highlights squares that changed state.

**Color convention**:
- **Yellow**: square emptied (piece left)
- **Cyan**: square occupied (piece arrived)
- **Gray**: no change

In [ ]:
if len(images) >= 2:
    occ_img_a = draw_occupancy_image(gt_a)
    occ_img_b = draw_occupancy_image(gt_b)
    diff_img = draw_move_diff(gt_a, gt_b)

    fig, axes = plt.subplots(2, 3, figsize=(15, 10))

    axes[0, 0].imshow(cv2.cvtColor(warped_a, cv2.COLOR_BGR2RGB))
    axes[0, 0].set_title(f"Frame A ({img_a_path.stem})")
    axes[0, 0].axis("off")
    axes[0, 1].imshow(cv2.cvtColor(warped_b, cv2.COLOR_BGR2RGB))
    axes[0, 1].set_title(f"Frame B ({img_b_path.stem})")
    axes[0, 1].axis("off")
    axes[0, 2].axis("off")
    axes[0, 2].text(0.5, 0.5, f"{len(changes)} changes\ndetectadas",
                     ha="center", va="center", fontsize=16,
                     transform=axes[0, 2].transAxes)

    axes[1, 0].imshow(cv2.cvtColor(occ_img_a, cv2.COLOR_BGR2RGB))
    axes[1, 0].set_title("Occupancy A (GT)")
    axes[1, 0].axis("off")
    axes[1, 1].imshow(cv2.cvtColor(occ_img_b, cv2.COLOR_BGR2RGB))
    axes[1, 1].set_title("Occupancy B (GT)")
    axes[1, 1].axis("off")
    axes[1, 2].imshow(cv2.cvtColor(diff_img, cv2.COLOR_BGR2RGB))
    axes[1, 2].set_title("Differences (yellow=emptied, cyan=occupied)")
    axes[1, 2].axis("off")

    plt.suptitle("Move detection - temporal analysis", fontsize=14)
    plt.tight_layout()
    save_fig("12_move_detection", RUN_DIR)
    plt.show()

## 10.2. From Changes to Move Notation

`detect_moves` tells us *which* squares changed; to name the move we also need the
piece types. `classify_move` compares two `piece_map`s and infers the move —
including **captures, castling, en passant and promotion** — returning UCI plus a
short description.

The synthetic dataset has no continuous games (each image is an independent random
position), so we validate the logic on **controlled** before/after positions, then
show what `classify_move` reports for two unrelated dataset frames.

In [ ]:
from src.chess import classify_move

# Standard starting position as a piece_map
_back = ["rook", "knight", "bishop", "queen", "king", "bishop", "knight", "rook"]
_start = {}
for _i, _f in enumerate("ABCDEFGH"):
    _start[f"{_f}8"] = _back[_i] + "_b"; _start[f"{_f}7"] = "pawn_b"
    _start[f"{_f}2"] = "pawn_w";         _start[f"{_f}1"] = _back[_i] + "_w"

_examples = {
    "1.e4 (lance simples)": (_start, {**{k: v for k, v in _start.items() if k != "E2"}, "E4": "pawn_w"}),
    "captura exd5":         ({"E4": "pawn_w", "D5": "pawn_b"}, {"D5": "pawn_w"}),
    "roque curto O-O":      ({"E1": "king_w", "H1": "rook_w"}, {"G1": "king_w", "F1": "rook_w"}),
    "promocao e8=Q":        ({"E7": "pawn_w"}, {"E8": "queen_w"}),
    "en passant e5xd6":     ({"E5": "pawn_w", "D5": "pawn_b"}, {"D6": "pawn_w"}),
}
for _name, (_b, _a) in _examples.items():
    _mv = classify_move(_b, _a)
    print(f"{_name:22s} -> uci={str(_mv['uci']):6s} | {_mv['description']}")

# Two unrelated dataset frames (no real move between them)
print()
if "ann_a" in dir() and "ann_b" in dir():
    _mv = classify_move(ann_a["config"], ann_b["config"])
    print("frames A vs B do dataset ->", _mv["type"], "|", _mv["description"])

# 11. Current State and Next Steps

| Step | Description | Status |
|---|---|---|
| **1. Board reading** | Detect, correct perspective, segment 8x8 | Completed |
| **2. Occupancy detection** | Classify empty/occupied by classical features | Completed |
| **3. Color classification** | Identify light/dark piece via HSV | Completed |
| **4. Type identification** | ResNet-34 transfer learning - 12 classes | Completed |
| **5. FEN reconstruction** | `board_to_fen` - all six FEN fields, validated | Completed |
| **6. Move notation** | `classify_move` - UCI + captures/castling/en passant/promotion | Completed |

## What works
- Full classical geometry + occupancy + color + ResNet-34 type -> `piece_map`
- End-to-end automatic read (auto corners -> occupancy -> type) -> **FEN**
- Complete FEN with all fields (`board_to_fen`), plus a castling-rights guess
  from the position (`castling_rights_from_position`)
- Move inference (`classify_move`): quiet, capture, castling, en passant, promotion

# 12. Known Limitations

- **Uniform material**: wood pieces and board → low contrast (challenges classical occupancy detection)
- **3D projection**: tall pieces "bleed" into neighbouring cells in the rectified image
- **180° ambiguity**: chessboard pattern is symmetric under 180° rotation - square-color detection only resolves 2 of the 4 rotations
- **Fixed thresholds**: calibrated for the synthetic dataset, may not generalise to real images
- **Synthetic dataset**: CNN trained on renders may not generalize to real board photos (domain shift)

# 13. Saving Metrics

In [ ]:
from src.utils import save_metrics

metrics = {
    "n_images_evaluated": len(results),
    "mean_accuracy": float(mean_acc),
    "mean_f1": float(mean_f1),
    "per_image": results,
}
save_metrics(metrics, RUN_DIR)
print("Metrics saved.")